# 02 · Trend deep dive

**What this notebook answers.** Five questions that are harder than "who owns this
keyword", and that only make sense over time:

1. **Colour identity drift** — has a colour's *toolbox* changed? Does red in 2024 use a
   different mix of mechanics than red in 2004?
2. **Pie-break tracking** — has a mechanic leaked out of the colour that used to own it?
3. **Rarity migration** — is a mechanic being pushed down to common, or rare-gated?
4. **Type-line crossover** — has a mechanic moved off creatures onto spells or enchantments?
5. **Complexity proxy** — are cards getting wordier, and is that even across colours?

Notebook 01 covers the foundations: the two weighting schemes, share vs penetration, and
where the tables come from. Read it first — the metrics here build on that vocabulary.
Magic knowledge is assumed; statistics are explained as they come up.

**Why this notebook groups sets.** The setup cell opens the connection with
`PeriodGroupConfig(mode="rolling_sets", group_size=5)` instead of the default per-set
grouping. Every measure below is a ratio computed *within* a period — a mix, a mean, a
similarity — so a period holding three relevant cards produces meaningless swings. Grouping
five consecutive sets buys enough cards per period for the numbers to be stable. It costs
nothing to change: grouping happens at query time, not at build time.

Requires `mtg-analysis build` to have run.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import polars as pl

from mtg_analysis.analysis.color_drift import color_drift, keyword_vectors, plot_color_drift
from mtg_analysis.analysis.complexity import complexity_by_color, plot_complexity
from mtg_analysis.analysis.db import get_connection
from mtg_analysis.analysis.pie_break import pie_break, plot_pie_break
from mtg_analysis.analysis.rarity_migration import plot_rarity_migration, rarity_migration
from mtg_analysis.analysis.type_crossover import plot_type_crossover, type_crossover
from mtg_analysis.config import PeriodGroupConfig, load_config

config = load_config("../config/config.yaml")
# Grouping consecutive sets smooths the small-set noise these views are sensitive to.
con = get_connection("../" / config.paths.processed_dir,
                     PeriodGroupConfig(mode="rolling_sets", group_size=5))

## 1 · Colour identity drift

**What it measures.** Whether a colour's *mix* of keywords has shifted. Not whether red
prints more cards, and not whether red gained a specific mechanic — whether the overall
shape of red's toolbox today resembles the shape it had before.

**Where the data comes from.** `keyword_facts`, `paper_only`, grouped by
`(period, colour, keyword)`. For each colour in each period, the keyword weights are
normalised into a frequency vector that sums to 1:

$$\text{freq}(K) = \frac{\sum w(K)}{\sum_{k} \sum w(k)}$$

So red in one period becomes something like *(Haste 0.31, First strike 0.18, Trample 0.15,
Menace 0.12, …)* — "of everything keyword-ish red did this period, this is how it split".
Because each vector is normalised, this measures a change in **mix**, not in volume: a
period where red printed half as many cards but in the same proportions scores as no drift.

**The calculation.** Two vectors are compared with **cosine similarity** — the standard way
to ask "do these two lists of proportions point in the same direction", ignoring their
overall size:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert\mathbf{a}\rVert \, \lVert\mathbf{b}\rVert}$$

1.0 means an identical mix; lower means the toolbox moved. (It cannot go below 0 here,
since no frequency is negative.)

**Worked example** with a two-keyword world:

- Era A red = (Haste 0.8, Trample 0.2)
- Era B red = (Haste 0.5, Trample 0.5)

$$\mathbf{a}\cdot\mathbf{b} = 0.8(0.5) + 0.2(0.5) = 0.5$$
$$\lVert\mathbf{a}\rVert = \sqrt{0.8^2 + 0.2^2} = \sqrt{0.68} \approx 0.8246 \qquad \lVert\mathbf{b}\rVert = \sqrt{0.5^2+0.5^2} \approx 0.7071$$
$$\cos = \frac{0.5}{0.8246 \times 0.7071} \approx \mathbf{0.86}$$

A noticeable but not dramatic shift — red still leans on Haste, just less exclusively.

**The two panels.**

- `cosine_vs_baseline` (top) compares each period against the colour's **all-time** mix.
  This is the one to read for long-run drift: a line that slides downward means today's
  red looks less and less like red-across-all-of-history.
- `cosine_vs_previous` (bottom) compares each period against the one before it. Spiky by
  nature; useful for spotting a single disruptive block rather than a trend.

**Reading notes and caveats.**

- `min_keyword_weight=5.0` (the default) drops periods where the colour had too few
  keyword appearances to mean anything. Dropping a period also **breaks the chain**, so
  `cosine_vs_previous` is null for the period after a gap — that is why the bottom line has
  holes.
- Keywords added recently (Ward, say) inflate drift simply by existing. A falling line is
  evidence of *change*, not of design quality.
- The summary table under the chart ranks colours by mean baseline similarity: lowest =
  most drifted overall.

In [ ]:
drift = color_drift(con)
fig, axes = plt.subplots(2, 1, figsize=(11, 9))
plot_color_drift(drift, metric="cosine_vs_baseline", ax=axes[0])
plot_color_drift(drift, metric="cosine_vs_previous", ax=axes[1])
fig.tight_layout()
plt.show()
drift.group_by("color").agg(pl.col("cosine_vs_baseline").mean()).sort("cosine_vs_baseline")

### Which keywords drove the shift?

The drift number tells you *that* a colour moved, never *how*. This cell dumps the
underlying frequency vectors — one row per (period, colour, keyword) with its `weight` and
its normalised `freq` — so you can see which mechanics actually grew or shrank inside a
colour's mix. Change the colour filter to investigate whichever line looked interesting
above.

In [ ]:
# The raw vectors, if you want to see which keywords drove a shift.
keyword_vectors(con).filter(pl.col("color") == "R").sort("freq", descending=True).head(10)

## 2 · Pie-break tracking

**What it measures.** Whether a mechanic has leaked out of the colour that historically
owned it — the "black is getting card draw" / "green is getting reach-adjacent flying"
question, made measurable.

**Where the data comes from.** `keyword_facts`, `paper_only`, grouped by
`(period, colour)` for the one keyword you pass. Each period's colour shares are computed
the same way as the heatmap in notebook 01, then the dominant colour is picked and
everything else is treated as a challenger.

**The calculation.** Two steps.

1. **Find the historical owner.** Take the earliest period holding at least
   `min_period_weight` (default `3.0`) of the keyword — enough cards to be a real signal,
   not one printing — and the colour with the top share there is the `original_dominant_color`.
2. **Track the leak.** For every period:

$$\text{challenger share}(P) = 1 - \text{share}_{\text{dominant}}(P)$$

In words: *how much of this keyword is now in colours other than the one that invented it?*

**Worked example.** Suppose the earliest qualifying set splits a keyword white 70% / red
30% — white is the dominant colour, challenger share 0.30. Twenty sets later the split is
white 40% / red 35% / black 25%: challenger share is now 0.60. The mechanic has gone from
mostly-white to mostly-not-white.

**How to read the chart.** One line per colour showing its share of the keyword over time;
the title names the colour that originally owned it. Lines converging toward each other =
the mechanic has become pan-colour. The dominant line falling while others rise = a genuine
pie break.

**The misreading to avoid.** A rising challenger share does **not** mean the dominant colour
is printing less of the mechanic — only that its *slice* shrank. The other colours getting
more, with the owner unchanged, produces exactly the same line. To tell those apart, look
at the owner's penetration rate in notebook 01.

In [ ]:
breaks = pie_break(con, "Double strike")
plot_pie_break(breaks)
plt.show()
breaks.select("period", "color", "share", "challenger_share").tail(10)

## 3 · Rarity migration

**What it measures.** Whether a mechanic is becoming *core* (pushed down to common, where
every limited deck sees it) or *premium* (rare-gated, saved for splashy cards).

**Where the data comes from.** `keyword_facts`, `paper_only`, grouped by period (pass
`by_color=True` to split by colour as well). Rarity is encoded ordinally:

$$\text{common} = 0, \quad \text{uncommon} = 1, \quad \text{rare} = 2, \quad \text{mythic} = 3$$

**The calculation.** A colour-weighted mean of that ordinal, per period:

$$\bar{r}(P) = \frac{\sum_i w_i \, r_i}{\sum_i w_i}$$

**Worked example.** A period where the keyword appears on two commons and one rare, all
mono-coloured (so each weight is 1.0):

$$\bar{r} = \frac{0 + 0 + 2}{3} \approx \mathbf{0.67}$$

Just above common — the mechanic is mostly a common-slot effect in that period.

**How to read the chart.** The y-axis is labelled with rarity names rather than numbers.
A line trending **up** means the mechanic is being reserved for higher rarities; **down**
means it is being pushed to common. Check `n_appearances` in the table before trusting a
swing — a period with four appearances can move the mean a whole rarity step.

**Two honest caveats.**

- The ordinal scale assumes the gap common→uncommon equals uncommon→rare, which is not
  really true of how Magic drafts play. Treat the line as a direction, not a distance.
- Rarities outside those four — `special`, `bonus` — are excluded from both the numerator
  and the denominator, so Timeshifted-style printings do not distort the mean.

In [ ]:
rarity = rarity_migration(con, "Haste")
plot_rarity_migration(rarity)
plt.show()
rarity.tail()

## 4 · Type-line crossover

**What it measures.** Where a keyword lives on the card pool. Haste starting as a
creature-only ability and spreading onto equipment, enchantments or instants is a real
design shift, and this is the chart that shows it.

**Where the data comes from.** `keyword_facts`, `paper_only`, grouped by
`(period, card type bucket)`. The bucket is derived from `type_line` by matching in a fixed
order:

```
Creature → Planeswalker → Instant/Sorcery → Enchantment → Artifact → Land → Other
```

**This ordering matters and changes how the bands read.** The *first* match wins, so
"Artifact Creature — Golem" counts as **Creature**, not Artifact, and "Enchantment
Creature — Bear" is also Creature. The bands answer "is this on a body or not" more than
"what is the card's dominant type".

**The calculation.** Share of the keyword's weighted appearances held by each type bucket
within a period:

$$\text{share}(T, P) = \frac{\sum w(T, P)}{\sum_{t} \sum w(t, P)}$$

Shares within a period sum to 1, which is why this is drawn as a stacked area.

**How to read the chart.** Bands stack to 100%; a band widening means that type took a
larger share of the keyword's appearances *that period*. Vertical position of the boundary
lines is what carries the signal, not the colour ramp (a single hue, stepped — these are
ordered parts of one whole, not independent categories).

**The misreading to avoid.** A growing band can mean more of that type — or the same
amount while everything else shrank. It is a share, so check the raw appearance counts in
the returned frame before saying "there are more of these now".

In [ ]:
crossover = type_crossover(con, "Haste")
plot_type_crossover(crossover)
plt.show()

## 5 · Complexity proxy

**What it measures.** A rough rules-bloat indicator: are cards getting wordier and
keyword-denser, and is that happening evenly across the colours?

**Where the data comes from.** `card_facts` — note, **all cards**, not just cards with
keywords, unlike every other chart in this notebook. Grouped by `(period, colour)`, using
two per-card fields computed at ingest: `oracle_text_length` (character count of the rules
text; multi-face cards have their faces joined with `//`) and `n_keywords` (how many formal
keywords the card has).

**The calculation.** Colour-weighted means, so a gold card contributes half to each of its
colours under fractional weighting rather than counting twice:

$$\overline{len}(P, C) = \frac{\sum_i w_i \cdot len_i}{\sum_i w_i} \qquad\qquad \overline{kw}(P, C) = \frac{\sum_i w_i \cdot kw_i}{\sum_i w_i}$$

**Worked example.** A period where red has a mono-red 40-character card and a red-white
100-character card: red's weighted mean text length is
$(1.0 \times 40 + 0.5 \times 100) / 1.5 = 90/1.5 = \mathbf{60}$ characters — not 70, because
the gold card is only half red's responsibility.

**How to read the charts.** Top panel is mean text length, bottom is mean keyword count,
both per colour over time. A colour sitting consistently above the others is printing
wordier cards than its neighbours that period; a general upward drift across all colours is
the rules-bloat signal.

**The caveat that matters.** Character count is a crude proxy for complexity. Reminder text
inflates it without adding rules; a terse card can be far more complex to play than a long
one; and keyword count *reduces* text length by design (that is what keywords are for), so
the two panels can disagree and both be right. Use it as a trend line, never as a
measurement of how hard cards are.

In [ ]:
complexity = complexity_by_color(con)
fig, axes = plt.subplots(2, 1, figsize=(11, 9))
plot_complexity(complexity, metric="mean_text_length", ax=axes[0])
plot_complexity(complexity, metric="mean_keywords", ax=axes[1])
fig.tight_layout()
plt.show()

---

# Write-up · what I found

Each of these views answers a narrower question than it first appears to. The prompts keep
the claim matched to the evidence.

### Claim

> *e.g. "Green's toolbox has drifted more than any other colour since 2015."*

### Evidence

| View | What it showed | Period grouping | Weighting |
|---|---|---|---|
| Colour drift (baseline / previous) | | | |
| Pie-break (dominant colour, challenger share) | | | |
| Rarity migration (direction, `n_appearances`) | | | |
| Type crossover (which band moved) | | | |
| Complexity (length / keyword count) | | | |

### Checks before believing it

- [ ] Enough cards per period — did `min_keyword_weight` / `min_period_weight` drop most of them?
- [ ] Drift measures **mix**, not volume — is the volume story in notebook 01 consistent?
- [ ] Pie-break: did the owner actually print less, or did others print more?
- [ ] Rarity: is the swing bigger than a handful of appearances could explain?
- [ ] Type crossover: share moved, or absolute count moved?
- [ ] Claim re-run at a different `group_size` — does it survive?

### Conclusion